# **Projet MLOps — Prédiction du défaut de crédit**

**Objectif** : Construire un modèle de scoring crédit qui estime la probabilité de défaut (PD) pour chaque client à partir de variables tabulaires (revenu, dette, ancienneté, score FICO, etc.).

Des prédictions fiables aident la banque à dimensionner le capital pour couvrir les pertes attendues et à stabiliser le risque.

## **Plan**

1. EDA rapide & préparation

2. Feature engineering léger (ratio dette/revenu)

3. Prétraitement dans un pipeline (imputation médiane, scaling seulement pour les modèles linéaires)

4. Entraînement de 3 modèles (Logistic, Decision Tree, Random Forest) + tracking MLflow

5. Sélection sur val_roc_auc, sérialisation du pipeline

6. Démo d’inférence avec Streamlit

## **1. Setup et Exploration (EDA)**

L'Analyse Exploratoire des Données (EDA) est la première étape concrète de l'analyse après le setup du projet et elle est cruciale pour le succès de notre modèle de risque de crédit. Elle fait partie de l'étape de Pré-traitement des données

On inspecte la structure du dataset, la qualité des variables et la distribution de la cible default.

Le dataset est entièrement numérique et propre ; on privilégiera AUC-ROC / Recall / F1 (jeu légèrement déséquilibré).

### **A. Chargement des Données**

In [ ]:
#Librairies de manipulation de données
import pandas as pd # type: ignore
import numpy as np # type: ignore

#Librairies de visualisation
import matplotlib.pyplot as plt # type: ignore
import seaborn as sns # type: ignore

#Librairies de Machine Learning et MLOps
import sklearn  # type: ignore
import mlflow # type: ignore

In [ ]:
df = pd.read_csv('Loan_Data.csv')

In [ ]:
pd.read_csv('Loan_Data.csv')

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

**Variation d'Échelle** : L'income (≈70000 en moyenne) et le fico_score (≈637) sont sur des échelles très différentes ce qui rend l'étape de mise à l'échelle (scaling) absolument nécessaire avant d'utiliser la Régression Logistique ou d'autres modèles sensibles aux échelles.

### **B. Compréhension et Analyse de la Variable Cible (y)**

Le cœur de ce projet est de prévoir le défaut de paiement. On calcule le taux de défaut (proportion de default = 1).

* AUC-ROC (Area Under the Receiver Operating Characteristic Curve) : La métrique la plus courante pour le risque de crédit. Elle mesure la capacité du modèle à classer correctement les défauts sur toute la plage de seuils.

* Recall (Rappel) : Mesure la proportion de défauts réels que votre modèle a correctement identifiés (vrais positifs / (vrais positifs + faux négatifs)). Essentiel pour la banque afin de ne pas manquer de mauvais payeurs.

* F1-Score : La moyenne harmonique de la précision (Precision) et du rappel (Recall).

In [ ]:
#Taux de défaut de paiement
taux_defaut = df['default'].mean() * 100
print(f"\nTaux de défaut de paiement: {taux_defaut:.2f}%")

**Taux de Défaut** : La moyenne de la variable default est de 0.1851. Cela signifie que le taux de défaut est de 18.51%. C'est un taux significatif qui confirme que nous devons utiliser des métriques robustes comme l'AUC-ROC ou le F1-Score

In [ ]:
#Visualisation de la distribution de la variable cible
plt.figure(figsize=(6, 4))
sns.countplot(x='default', data=df)
plt.title('Distribution de la variable cible (Défaut de paiement)')
plt.xticks([0, 1], ['Remboursé (0)', 'Défaut de paiement (1)'])
plt.show()

**Clients remboursés (0)** : Le graphique montre environ 8000 cas qui est la grande majorité.

**Défauts de paiement (1)** : Le graphique montre environ 1851 cas basé sur la moyenne de 0.1851 précédemment fournie correspondant à la barre à un peu moins de 2000.

Ce déséquilibre a des implications directes sur la manière d'évaluer nos modèles.

* FICO score plus bas chez les clients en défaut → fort pouvoir discriminant.
* Variables financières (dette/encours/revenu) pertinentes → on les conserve toutes.

In [ ]:
#Relation entre FICO Score et Défaut de paiement
plt.figure(figsize=(8, 6))
sns.boxplot(x='default', y='fico_score', data=df)
plt.title('Distribution du FICO Score selon le statut de Défaut de paiement')
plt.xticks([0, 1], ['Remboursé', 'Défaut de paiement'])
plt.show()

Le graphique montre que la Distribution du FICO Score selon le statut de Défaut de paiement confirme une relation fondamentale dans le risque de crédit :
* Remboursé (0) : Les clients qui ont remboursé leur prêt ont des scores FICO significativement plus élevés (autour de 650-660).
* Défaut de paiement (1) : Les clients en défaut de paiement ont des scores FICO plus bas (autour de 600).

Conclusion : La variable fico_score est un excellent prédicteur de la variable cible (default).

La Matrice de Corrélation est essentielle pour comprendre la relation de chaque variable avec la cible (default) et les relations entre les autres variables.

In [ ]:
#Matrice de corrélation
correlation_matrix = df.drop(columns=['customer_id']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matrice de Corrélation')
plt.show()

## **2. Pré-traitement et Feature Engineering**

Cette étape la préparation des données pour les modèles.

### **A. Gestion des Valeurs Manquantes**

Les données incomplètes peuvent fausser les modèles

In [ ]:
print(df.isnull().sum())

Puisqu'on a pas de valeurs manquantes ni de variables catégorielles à encoder on se concentre sur le feature engineering

### **B. Feature engineering & séparation X/y**

On crée une feature robuste et standard en scoring crédit :  

**Formule :**  
`debt_income_variable = total_debt_outstanding / (income + ε)`

Cette variable représente le niveau d’endettement relatif au revenu.  
> **Remarque :** on ne supprime **aucune variable financière**, contrairement à certains exemples.


In [ ]:
def load_data(path: str, mode: str = "safe"):
    df = pd.read_csv(path)

    #Feature engineering
    eps = 1e-5
    df["debt_income_variable"] = df["total_debt_outstanding"] / (df["income"] + eps)

    y = df["default"].astype(int)
    X = df.drop(columns=["default", "customer_id"], errors="ignore")

    #Listes de colonnes soupçonnées de fuite
    LEAKY_MIN = ["debt_income_variable", "credit_lines_outstanding", "loan_amt_outstanding"]
    LEAKY_STRICT = LEAKY_MIN + ["total_debt_outstanding"]

    if mode == "safe":
        X = X.drop(columns=LEAKY_MIN, errors="ignore")
    elif mode == "strict":
        X = X.drop(columns=LEAKY_STRICT, errors="ignore")

    return X, y

In [ ]:
from sklearn.model_selection import train_test_split

def make_splits(X, y, test_size=0.2, val_size=0.2, random_state=42):
    #Train / Temp (stratification pour conserver ~18.5% défaut)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )
    #Val / Test à 50/50 sur le reste, toujours stratifié
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
X, y = load_data("Loan_Data.csv", mode="strict")
X_train, X_val, X_test, y_train, y_val, y_test = make_splits(X, y)
X.head(), y.head()

> Les splits sont **stratifiés** sur `y` pour conserver le ~18,5% de défaut dans train/val/test.

## **3. Prétraitement dans le pipeline**

On intègre le **prétraitement directement dans le pipeline scikit-learn** :
- **Imputation médiane** (sécurité, même si pas de NA ici),
- **Standardisation** uniquement pour la **régression logistique** (modèle sensible à l’échelle).

> Cela évite la **fuite de données** et garantit la **reproductibilité** (entraînement & inférence).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
def numeric_pipeline(with_scaler: bool = True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if with_scaler:
        steps.append(("scaler", StandardScaler()))
    return Pipeline(steps)

In [ ]:
def build_model(kind: str):
    if kind == "logistic":
        clf = LogisticRegression(max_iter=200)
        pre = numeric_pipeline(with_scaler=True)    # scaling utile
    elif kind == "tree":
        clf = DecisionTreeClassifier(max_depth=6, random_state=42)
        pre = numeric_pipeline(with_scaler=False)   
    elif kind == "rf":
        clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
        pre = numeric_pipeline(with_scaler=False)   
    else:
        raise ValueError("kind must be in {logistic, tree, rf}")
    return Pipeline([("prep", pre), ("clf", clf)])

## **4. Entraînement des modèles & suivi MLflow**

On entraîne 3 modèles :
- **Logistic Regression** (baseline),
- **Decision Tree**,
- **Random Forest**.

Chaque exécution est trackée dans **MLflow** (paramètres, métriques, artefacts).  
On sélectionne le **meilleur modèle** sur **`val_roc_auc`** et on enregistre le pipeline complet (`best_model.joblib`).

In [ ]:
import os, joblib, mlflow
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, accuracy_score

In [ ]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    """Calcule les métriques clés à partir des probabilités de classe 1."""
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, y_prob),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
    }

In [ ]:
#1) Définir l'expérience MLflow
experiment_name = "CreditScoring"
mlflow.set_experiment(experiment_name)

> **Remarque — Sanity check & fuite d’information**  
> - Des AUC ≈ **1.00** en validation indiquent une **fuite d’information**.  
> - Un **audit univarié** a révélé des colonnes quasi déterministes :  
>   - `debt_income_variable` : **0.998**  
>   - `credit_lines_outstanding` : **0.996**  
>   - `total_debt_outstanding` : **0.963**  
> - Pour un cas d’usage **PD à l’origination**, ces variables peuvent capturer des informations **post-évènement** ou être construites **trop proches de la cible**.  
>   Nous entraînons donc avec un jeu de features **épuré (`mode="strict"`)**, en conservant des variables plausibles à l’instant de décision (ex. `fico_score`, `years_employed`, `income`, …).  
> - Après retrait, les performances deviennent **réalistes et généralisables**, validant l’**absence de fuite** -> CF: la suite.

In [ ]:
#2) Entraîner les candidats et logger
candidates = ["logistic", "tree", "rf"]
results = []
os.makedirs("models", exist_ok=True)

for kind in candidates:
    with mlflow.start_run(run_name=kind):
        #a) paramètres utiles à tracer
        mlflow.log_param("model", kind)
        mlflow.log_param("n_features", int(X.shape[1]))
        mlflow.log_param("features", ",".join(X.columns))

        #b) construction et entraînement du pipeline
        model = build_model(kind)
        model.fit(X_train, y_train)

        #c) éval: proba de la classe 1 (défaut)
        val_prob  = model.predict_proba(X_val)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        val_m  = compute_metrics(y_val,  val_prob)
        test_m = compute_metrics(y_test, test_prob)
        
        #Visualisations
        from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay
        import matplotlib.pyplot as plt

        #ROC (val)
        fig, ax = plt.subplots()
        RocCurveDisplay.from_predictions(y_val, val_prob, ax=ax)
        ax.set_title(f'ROC (val) — {kind}')
        mlflow.log_figure(fig, f'roc_val_{kind}.png'); plt.show(); plt.close(fig)

        #Precision–Recall (val)
        fig, ax = plt.subplots()
        PrecisionRecallDisplay.from_predictions(y_val, val_prob, ax=ax)
        ax.set_title(f'PR (val) — {kind}')
        mlflow.log_figure(fig, f'pr_val_{kind}.png'); plt.show(); plt.close(fig)

        #Matrice de confusion (val, thr=0.5)
        y_val_pred = (val_prob >= 0.5).astype(int)
        fig, ax = plt.subplots()
        ConfusionMatrixDisplay.from_predictions(y_val, y_val_pred, ax=ax)
        ax.set_title(f'CM (val, thr=0.5) — {kind}')
        mlflow.log_figure(fig, f'cm_val_thr05_{kind}.png'); plt.show(); plt.close(fig)


        #d) logging des métriques
        for k, v in val_m.items():
            mlflow.log_metric(f"val_{k}", float(v))
        for k, v in test_m.items():
            mlflow.log_metric(f"test_{k}", float(v))

        #e) garder sous la main pour la sélection
        results.append({
            "kind": kind,
            "model": model,
            "val_roc_auc": float(val_m["roc_auc"]),
            "test_metrics": test_m
        })


In [ ]:
#3) Sélection du meilleur modèle sur val_roc_auc
best = max(results, key=lambda d: d["val_roc_auc"])

In [ ]:
#4) sérialisation + log artefact dans un run dédié "selection"
best_path = "models/best_model.joblib"
with mlflow.start_run(run_name=f"selection_{best['kind']}"):
    joblib.dump({"pipeline": best["model"], "features": list(X.columns)}, best_path)
    mlflow.log_param("selected_model", best["kind"])
    mlflow.log_metric("selection_val_roc_auc", float(best["val_roc_auc"]))
    mlflow.log_artifact(best_path)

print(f"Meilleur modèle: {best['kind']} | val_roc_auc={best['val_roc_auc']:.4f}")
print("Test metrics:", best["test_metrics"])
print(f"Pipeline sérialisé → {best_path}")

## **5. Choix du seuil métier & sauvegarde**

**Pourquoi ?**  
À seuil 0.5, la précision est correcte mais le rappel est faible (jeu déséquilibré).  
On choisit un **seuil de décision** sur la **validation** (ex. celui qui **maximise le F1**), puis on **sauvegarde** ce seuil pour l’app.

**Ce que fait la cellule code suivante :**  
- calcule le seuil optimal sur *validation*,  
- affiche le **report** sur *test* à ce seuil,  
- écrit `models/serving_config.json` avec `decision_threshold` et `selected_model`.

> Résultat attendu : un compromis précision/rappel plus utile métier qu’un seuil fixe 0.5.

In [ ]:
#Choix d'un seuil qui maximise le F1 sur la validation, puis sauvegarde pour l'app
import numpy as np, json
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

best_model = best["model"]  #issu de l'étape 4
val_prob  = best_model.predict_proba(X_val)[:, 1]
test_prob = best_model.predict_proba(X_test)[:, 1]

ths = np.linspace(0, 1, 201)
scores = []
for t in ths:
    yv = (val_prob >= t).astype(int)
    scores.append((t,
                   precision_score(y_val, yv, zero_division=0),
                   recall_score(y_val, yv),
                   f1_score(y_val, yv)))

thr_star, p_val, r_val, f1_val = max(scores, key=lambda x: x[3])
print(f"Seuil choisi (max F1 sur val): {thr_star:.2f} | P={p_val:.3f} R={r_val:.3f} F1={f1_val:.3f}")

#Report test au seuil choisi
yt = (test_prob >= thr_star).astype(int)
print(classification_report(y_test, yt, digits=3))
ConfusionMatrixDisplay.from_predictions(y_test, yt); plt.title(f'CM (test, thr={thr_star:.2f})'); plt.show()

#Sauvegarde pour l'app Streamlit
cfg = {"decision_threshold": float(thr_star), "selected_model": best["kind"]}
with open("models/serving_config.json", "w") as f:
    json.dump(cfg, f, indent=2)
print("Config écrite -> models/serving_config.json")

## **6.Smoke test d’inférence (pipeline sérialisé)**

**Objectif :** vérifier que le pipeline `models/best_model.joblib` et la config  
`models/serving_config.json` fonctionnent bien **hors entraînement**.

**Ce que fait la cellule code suivante :**  
- charge le pipeline + la liste des features,  
- lit le seuil sauvegardé,  
- effectue quelques prédictions sur 5 échantillons *test* et affiche proba + décision.

> Si tout est OK, on peut passer à la démo Streamlit.

In [ ]:
import joblib, numpy as np, pandas as pd, json

obj = joblib.load("models/best_model.joblib")
pipe, feat_names = obj["pipeline"], obj["features"]

with open("models/serving_config.json") as f:
    thr = json.load(f).get("decision_threshold", 0.5)

# 5 échantillons de test
X_sample = X_test.iloc[:5][feat_names]
proba = pipe.predict_proba(X_sample)[:, 1]
pred  = (proba >= thr).astype(int)

display(pd.DataFrame({
    "proba_default": proba,
    "pred@thr": pred,
}, index=X_sample.index))

In [ ]:
#1) Afficher le seuil réellement utilisé et le % de positifs prédit sur TOUT le test
print("Seuil utilisé:", thr)
test_prob = pipe.predict_proba(X_test[feat_names])[:, 1]
y_pred = (test_prob >= thr).astype(int)
print("Taux prédits positifs:", y_pred.mean().round(3), " | Taux réel défaut:", y_test.mean().round(3))

In [ ]:
#2) Voir la distribution des probabilités (test)
import matplotlib.pyplot as plt, numpy as np, pandas as pd
pd.Series(test_prob).hist(bins=30); plt.axvline(thr, linestyle="--"); plt.title("Distribution des probabilités (test)"); plt.show()

## **7. Le Déploiement de l'application**

Le Déploiement de votre meilleur modèle (la Régression Logistique) sur une application web via un pipeline CI/CD conformément aux exigences du projet.

### **A. Création de l'Application Web avec streamlit**

Pour que votre modèle fonctionne correctement, l'application doit effectuer trois actions indispensables sur les données entrées par l'utilisateur :

Charger le Modèle Vainqueur : Récupérer le modèle de Régression Logistique et le StandardScaler (qui contient les moyennes et écarts-types appris sur X_train).

Feature Engineering : Calculer la variable debt_income_variable (DTI) à partir des entrées total_debt_outstanding et income du nouveau client.

Scaling : Appliquer le StandardScaler chargé sur les nouvelles données du client.

### **B. Pipeline CI/CD**

L'étape finale sera la mise en place d'un pipeline pour : 